# SGLang 连续批处理功能使用教程

**定位：** 详解 SGLang 的连续批处理（Continuous Batching）机制，通过实际并发实验对比串行与并发的性能差异。

**服务端启动命令（默认示例）：**

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

> 本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台

---

## 硬件与环境要求

| 项目 | 最低要求 |
|------|----------|
| GPU | NVIDIA GPU（支持 CUDA） |
| 显存（VRAM） | ≥ 6 GB（推荐 8 GB 以上） |
| 驱动 | NVIDIA Driver ≥ 525 |
| CUDA | CUDA ≥ 12.1 |
| Python | ≥ 3.9 |
| 操作系统 | Linux / WSL2 推荐 |

## 依赖安装

```bash
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和依赖就绪
2. 运行「启动 SGLang 服务」单元格
3. 依次运行并发请求和串行请求的对比实验
4. 查看吞吐量对比结果
5. 最后运行「清理」单元格释放资源

---

In [ ]:
# ============================================================
# 环境检查 —— 确认 GPU、PyTorch 和 SGLang 是否就绪
# ============================================================

import sys  # 导入系统模块

print("Python 版本：", sys.version)  # 打印 Python 版本

# ---------- 检查 PyTorch ----------
try:  # 尝试导入 PyTorch
    import torch  # 导入 PyTorch 库
    print("PyTorch 版本：", torch.__version__)  # 打印 PyTorch 版本
    print("CUDA 是否可用：", torch.cuda.is_available())  # 检查 CUDA
    if torch.cuda.is_available():  # 如果 CUDA 可用
        print("GPU 名称：", torch.cuda.get_device_name(0))  # 打印 GPU 名称
except ImportError:  # 如果未安装
    print("PyTorch 未安装")  # 提示未安装

# ---------- 检查 SGLang ----------
try:  # 尝试导入 SGLang
    import sglang  # 导入 sglang
    print("SGLang 版本：", sglang.__version__)  # 打印版本
except ImportError:  # 如果未安装
    print("SGLang 未安装")  # 提示未安装

# ---------- 检查 OpenAI 客户端 ----------
try:  # 尝试导入 OpenAI
    import openai  # 导入 openai 库
    print("OpenAI 版本：", openai.__version__)  # 打印版本
except ImportError:  # 如果未安装
    print("OpenAI 未安装")  # 提示未安装

print("\n环境检查完成")  # 打印完成提示

In [ ]:
# ============================================================
# 可选安装 —— 如依赖缺失则取消注释执行
# ============================================================

# !pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"  # 安装 SGLang 全部依赖及 OpenAI 客户端和 requests 库

## 什么是连续批处理（Continuous Batching）？

### 传统静态批处理（Static Batching）

- 一个批次（batch）中的所有请求必须**同时开始、同时结束**
- 如果某个请求只需生成 10 个 token，而另一个需要 200 个 token，短请求必须等长请求完成后才能返回
- GPU 在短请求完成后出现空闲浪费

### 连续批处理（Continuous Batching）

- 请求可以**随时加入或离开**正在运行的批次
- 短请求完成后立即返回结果，不必等待同一批次的其他请求
- 腾出的 GPU 资源立刻被新请求使用

### 核心优势

```
静态批处理:              连续批处理:

请求A ████████████        请求A ████████████
请求B ████------->等待     请求B ████ (完成即返回)
请求C ██████----->等待     请求C ██████ (完成即返回)
      所有请求等最长的           各请求独立完成
```

SGLang **默认启用**连续批处理，无需额外配置。

---

## 启动 SGLang 服务

In [ ]:
# ============================================================
# 启动 SGLang 服务 —— 后台启动并等待服务就绪
# ============================================================

import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求库

# ---------- 定义启动命令 ----------
cmd = [  # 定义 SGLang 启动命令列表
    "python", "-m", "sglang.launch_server",  # 使用 Python 模块方式启动
    "--model-path", "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型路径
    "--host", "127.0.0.1",  # 绑定本地地址
    "--port", "30000",  # 指定端口号
]

print("启动命令：")  # 打印命令标题
print(" ".join(cmd))  # 打印拼接后的命令

# ---------- 后台启动服务 ----------
print("\n正在启动 SGLang 服务...")  # 打印启动提示
server_process = subprocess.Popen(  # 以子进程方式启动服务
    cmd,  # 传入命令列表
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE  # 捕获标准错误
)

# ---------- 等待服务就绪 ----------
max_wait = 120  # 最大等待时间为 120 秒
start_time = time.time()  # 记录开始时间
server_ready = False  # 初始化就绪标志

while time.time() - start_time < max_wait:  # 在最大等待时间内循环
    try:  # 尝试健康检查
        resp = requests.get("http://127.0.0.1:30000/health", timeout=3)  # 向 /health 端点发送请求
        if resp.status_code == 200:  # 如果返回 200
            server_ready = True  # 标记就绪
            break  # 跳出循环
    except requests.ConnectionError:  # 如果连接失败
        pass  # 继续等待
    time.sleep(3)  # 每 3 秒检查一次

if server_ready:  # 如果服务就绪
    elapsed = time.time() - start_time  # 计算耗时
    print(f"SGLang 服务启动成功！耗时 {elapsed:.1f} 秒")  # 打印成功信息
else:  # 如果启动超时
    print(f"服务在 {max_wait} 秒内未就绪，请检查日志")  # 打印超时提示

## 模拟并发请求场景

我们将创建 **10 个请求**，每个请求要求生成不同长度的输出：
- 短请求（max_tokens=20）：模拟简短问答
- 中等请求（max_tokens=80）：模拟一般回答
- 长请求（max_tokens=200）：模拟详细回答

通过并发提交，观察连续批处理如何让短请求更快返回。

In [ ]:
# ============================================================
# 模拟并发请求场景 —— 创建不同输出长度的请求列表
# ============================================================

from openai import OpenAI  # 从 openai 库导入 OpenAI 客户端类
import threading  # 导入线程模块
import time  # 导入时间模块

# ---------- 创建 OpenAI 兼容客户端 ----------
client = OpenAI(  # 实例化 OpenAI 客户端
    base_url="http://127.0.0.1:30000/v1",  # 指向本地 SGLang 服务
    api_key="EMPTY"  # SGLang 不需要真实 API Key
)

# ---------- 定义不同长度的请求 ----------
task_configs = [  # 定义任务配置列表
    {"prompt": "用一句话说明什么是 Python", "max_tokens": 20, "label": "短-1"},  # 短请求 1
    {"prompt": "详细解释什么是机器学习，包括常见算法和应用场景", "max_tokens": 200, "label": "长-1"},  # 长请求 1
    {"prompt": "什么是 GPU", "max_tokens": 20, "label": "短-2"},  # 短请求 2
    {"prompt": "解释 HTTP 协议的工作原理", "max_tokens": 80, "label": "中-1"},  # 中等请求 1
    {"prompt": "用一个词描述太阳", "max_tokens": 20, "label": "短-3"},  # 短请求 3
    {"prompt": "详细介绍深度学习的发展历史和主要里程碑", "max_tokens": 200, "label": "长-2"},  # 长请求 2
    {"prompt": "什么是 API", "max_tokens": 20, "label": "短-4"},  # 短请求 4
    {"prompt": "解释操作系统中进程和线程的区别", "max_tokens": 80, "label": "中-2"},  # 中等请求 2
    {"prompt": "什么是数据库", "max_tokens": 20, "label": "短-5"},  # 短请求 5
    {"prompt": "详细说明 Transformer 架构的设计原理和各组件功能", "max_tokens": 200, "label": "长-3"},  # 长请求 3
]

print(f"共创建 {len(task_configs)} 个请求：")  # 打印请求总数
print(f"  短请求（max_tokens=20） ：5 个")  # 打印短请求数量
print(f"  中等请求（max_tokens=80）：2 个")  # 打印中等请求数量
print(f"  长请求（max_tokens=200）：3 个")  # 打印长请求数量
print("\n请求列表：")  # 打印请求列表标题
for i, cfg in enumerate(task_configs):  # 遍历每个请求配置
    print(f"  [{i+1:2d}] {cfg['label']:5s} | max_tokens={cfg['max_tokens']:3d} | {cfg['prompt']}")  # 打印请求详情

## 观察连续批处理效果

将 10 个请求**同时**提交给 SGLang 服务，记录每个请求的完成时间。

预期结果：短请求应当**先于**长请求完成（不被阻塞）。

In [ ]:
# ============================================================
# 观察连续批处理效果 —— 并发提交请求并记录各请求完成时间
# ============================================================

import threading  # 导入线程模块
import time  # 导入时间模块

concurrent_results = []  # 初始化并发结果列表
result_lock = threading.Lock()  # 创建线程锁保护结果列表

def send_request(task_cfg, global_start):  # 定义发送单个请求的函数
    """发送一个 Chat Completion 请求并记录耗时。"""
    req_start = time.time()  # 记录请求开始时间
    try:  # 尝试发送请求
        response = client.chat.completions.create(  # 调用 Chat Completion API
            model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
            messages=[{"role": "user", "content": task_cfg["prompt"]}],  # 设置用户消息
            max_tokens=task_cfg["max_tokens"],  # 设置最大生成 token 数
            temperature=0.7,  # 设置温度参数
        )
        req_end = time.time()  # 记录请求结束时间
        duration = req_end - req_start  # 计算请求耗时
        relative_end = req_end - global_start  # 计算相对完成时间
        tokens = response.usage.completion_tokens  # 获取实际生成的 token 数
        with result_lock:  # 获取线程锁
            concurrent_results.append({  # 将结果添加到列表
                "label": task_cfg["label"],  # 请求标签
                "max_tokens": task_cfg["max_tokens"],  # 最大 token 数
                "actual_tokens": tokens,  # 实际生成的 token 数
                "duration": duration,  # 请求耗时
                "relative_end": relative_end,  # 相对完成时间
                "success": True,  # 标记成功
            })
    except Exception as e:  # 如果请求失败
        with result_lock:  # 获取线程锁
            concurrent_results.append({  # 添加失败记录
                "label": task_cfg["label"],  # 请求标签
                "max_tokens": task_cfg["max_tokens"],  # 最大 token 数
                "actual_tokens": 0,  # 无生成 token
                "duration": time.time() - req_start,  # 记录耗时
                "relative_end": time.time() - global_start,  # 相对完成时间
                "success": False,  # 标记失败
                "error": str(e),  # 记录错误信息
            })

# ---------- 并发提交所有请求 ----------
print("开始并发提交 10 个请求...")  # 打印开始提示
print("=" * 60)  # 打印分隔线

threads = []  # 初始化线程列表
global_start = time.time()  # 记录全局开始时间

for cfg in task_configs:  # 遍历所有任务配置
    t = threading.Thread(target=send_request, args=(cfg, global_start))  # 为每个请求创建线程
    threads.append(t)  # 添加到线程列表
    t.start()  # 启动线程

for t in threads:  # 遍历所有线程
    t.join()  # 等待所有线程完成

concurrent_total_time = time.time() - global_start  # 计算并发总耗时

# ---------- 按完成时间排序并打印结果 ----------
concurrent_results.sort(key=lambda x: x["relative_end"])  # 按相对完成时间排序

print(f"\n并发请求完成！总耗时: {concurrent_total_time:.2f} 秒")  # 打印总耗时
print("\n请求完成时间线（按完成顺序）：")  # 打印时间线标题
print(f"{'序号':>4s} | {'标签':>5s} | {'max_tokens':>10s} | {'实际tokens':>8s} | {'耗时(秒)':>8s} | {'完成时刻':>8s} | 时间线")  # 打印表头
print("-" * 85)  # 打印分隔线

for i, r in enumerate(concurrent_results):  # 遍历排序后的结果
    bar_len = int(r["relative_end"] / concurrent_total_time * 30)  # 计算时间线长度
    bar = "#" * bar_len  # 生成时间线字符串
    status = "OK" if r["success"] else "FAIL"  # 设置状态标记
    print(f" {status:>4s} {i+1:2d} | {r['label']:>5s} | {r['max_tokens']:>10d} | {r['actual_tokens']:>8d} | {r['duration']:>8.2f} | {r['relative_end']:>7.2f}s | {bar}")  # 打印详情

## 对比：逐条串行处理

将同样的 10 个请求**逐条顺序发送**，观察总耗时差异。

In [ ]:
# ============================================================
# 对比：逐条串行处理 —— 顺序发送同样的请求
# ============================================================

import time  # 导入时间模块

sequential_results = []  # 初始化串行结果列表

print("开始串行发送 10 个请求...")  # 打印开始提示
print("=" * 60)  # 打印分隔线

seq_global_start = time.time()  # 记录串行测试的全局开始时间

for i, cfg in enumerate(task_configs):  # 遍历所有任务配置
    req_start = time.time()  # 记录单个请求的开始时间
    try:  # 尝试发送请求
        response = client.chat.completions.create(  # 调用 Chat Completion API
            model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型
            messages=[{"role": "user", "content": cfg["prompt"]}],  # 设置消息
            max_tokens=cfg["max_tokens"],  # 设置最大 token 数
            temperature=0.7,  # 设置温度参数
        )
        req_end = time.time()  # 记录请求结束时间
        duration = req_end - req_start  # 计算请求耗时
        tokens = response.usage.completion_tokens  # 获取实际生成 token 数
        sequential_results.append({  # 将结果添加到列表
            "label": cfg["label"],  # 请求标签
            "max_tokens": cfg["max_tokens"],  # 最大 token 数
            "actual_tokens": tokens,  # 实际 token 数
            "duration": duration,  # 请求耗时
            "success": True,  # 标记成功
        })
        print(f"  [{i+1:2d}/10] {cfg['label']:>5s} | 耗时 {duration:.2f}s | 生成 {tokens} tokens")  # 打印结果
    except Exception as e:  # 如果请求失败
        duration = time.time() - req_start  # 计算耗时
        sequential_results.append({  # 添加失败记录
            "label": cfg["label"],  # 请求标签
            "max_tokens": cfg["max_tokens"],  # 最大 token 数
            "actual_tokens": 0,  # 无 token
            "duration": duration,  # 耗时
            "success": False,  # 标记失败
        })
        print(f"  [{i+1:2d}/10] {cfg['label']:>5s} | 失败: {e}")  # 打印失败

sequential_total_time = time.time() - seq_global_start  # 计算串行总耗时
print(f"\n串行处理完成！总耗时: {sequential_total_time:.2f} 秒")  # 打印总耗时

## 吞吐量对比

In [ ]:
# ============================================================
# 吞吐量对比 —— 串行 vs 并发的性能差异分析
# ============================================================

print("=" * 60)  # 打印边框
print("        连续批处理 vs 串行处理 性能对比")  # 打印标题
print("=" * 60)  # 打印边框

# ---------- 计算统计指标 ----------
concurrent_tokens = sum(r["actual_tokens"] for r in concurrent_results if r["success"])  # 统计并发模式总 token
sequential_tokens = sum(r["actual_tokens"] for r in sequential_results if r["success"])  # 统计串行模式总 token

concurrent_success = sum(1 for r in concurrent_results if r["success"])  # 统计并发成功数
sequential_success = sum(1 for r in sequential_results if r["success"])  # 统计串行成功数

# ---------- 打印对比表 ----------
print(f"\n{'指标':20s} | {'并发（连续批处理）':>18s} | {'串行':>18s}")  # 打印表头
print("-" * 65)  # 打印分隔线
print(f"{'总耗时':20s} | {concurrent_total_time:>16.2f}s | {sequential_total_time:>16.2f}s")  # 打印总耗时
print(f"{'成功请求数':20s} | {concurrent_success:>17d} | {sequential_success:>17d}")  # 打印成功数
print(f"{'总生成 tokens':20s} | {concurrent_tokens:>17d} | {sequential_tokens:>17d}")  # 打印总 token

if concurrent_total_time > 0:  # 如果并发耗时大于 0
    concurrent_throughput = concurrent_tokens / concurrent_total_time  # 计算并发吞吐量
else:  # 耗时为 0
    concurrent_throughput = 0  # 吞吐量为 0

if sequential_total_time > 0:  # 如果串行耗时大于 0
    sequential_throughput = sequential_tokens / sequential_total_time  # 计算串行吞吐量
else:  # 耗时为 0
    sequential_throughput = 0  # 吞吐量为 0

print(f"{'吞吐量 (tokens/s)':20s} | {concurrent_throughput:>16.1f} | {sequential_throughput:>16.1f}")  # 打印吞吐量

# ---------- 计算加速比 ----------
if sequential_total_time > 0 and concurrent_total_time > 0:  # 如果两者耗时都大于 0
    speedup = sequential_total_time / concurrent_total_time  # 计算加速比
    print(f"{'加速比':20s} | {speedup:>16.2f}x | {'1.00x (基准)':>18s}")  # 打印加速比
    time_saved = sequential_total_time - concurrent_total_time  # 计算节省时间
    print(f"\n并发模式比串行模式快 {speedup:.2f} 倍，节省 {time_saved:.2f} 秒")  # 打印总结

# ---------- 短请求延迟对比 ----------
print("\n" + "=" * 60)  # 打印分隔线
print("短请求（max_tokens=20）延迟对比：")  # 打印短请求延迟标题
print("-" * 60)  # 打印分隔线

concurrent_short = [r for r in concurrent_results if r["max_tokens"] == 20 and r["success"]]  # 筛选并发短请求
sequential_short = [r for r in sequential_results if r["max_tokens"] == 20 and r["success"]]  # 筛选串行短请求

if concurrent_short:  # 如果有并发短请求结果
    avg_concurrent_short = sum(r["duration"] for r in concurrent_short) / len(concurrent_short)  # 计算平均耗时
    print(f"  并发模式平均耗时: {avg_concurrent_short:.3f}s")  # 打印并发平均耗时

if sequential_short:  # 如果有串行短请求结果
    avg_sequential_short = sum(r["duration"] for r in sequential_short) / len(sequential_short)  # 计算平均耗时
    print(f"  串行模式平均耗时: {avg_sequential_short:.3f}s")  # 打印串行平均耗时

print("\n连续批处理的关键优势：短请求不会被长请求阻塞！")  # 打印核心结论

## 连续批处理的优势总结

通过以上实验，我们可以看到连续批处理带来的显著优势：

| 优势 | 说明 |
|------|------|
| **更高的 GPU 利用率** | GPU 不会因等待长请求而空闲，始终保持高效计算 |
| **更低的平均延迟** | 短请求可以快速完成，不被长请求阻塞 |
| **更好的吞吐量** | 单位时间内处理更多请求和生成更多 token |
| **短请求不被长请求阻塞** | 每个请求独立完成，用户体验更好 |

### SGLang 中的连续批处理

- SGLang **默认启用**连续批处理，无需额外参数
- 配合 **RadixAttention**（前缀缓存），可以进一步提升多请求场景的效率
- 服务端自动管理批次调度，用户只需正常发送请求即可

---

In [ ]:
# ============================================================
# 清理 —— 停止 SGLang 服务并释放资源
# ============================================================

import subprocess  # 导入子进程模块

# ---------- 停止 SGLang 服务进程 ----------
if 'server_process' in dir() and server_process.poll() is None:  # 如果服务进程仍在运行
    server_process.terminate()  # 发送终止信号
    server_process.wait(timeout=10)  # 等待进程结束
    print("SGLang 服务已停止")  # 打印停止提示
else:  # 如果进程已结束
    print("SGLang 服务未在运行")  # 打印提示

# ---------- 清理 GPU 缓存 ----------
try:  # 尝试清理 GPU 缓存
    import torch  # 导入 PyTorch
    if torch.cuda.is_available():  # 如果 CUDA 可用
        torch.cuda.empty_cache()  # 清空 GPU 缓存
        print("GPU 缓存已清理")  # 打印清理完成
except ImportError:  # 如果未安装
    pass  # 跳过

print("\n清理完成")  # 打印清理完成信息

## 常见问题排查

### 问题 1：并发请求没有加速效果

**现象：** 并发总耗时与串行总耗时相差不大，看不到明显加速。

**排查步骤：**
1. 检查 GPU 是否已处于满负荷状态（此时增加并发不会提升速度）
2. 减少并发请求数量（如从 10 降到 5），避免超过 GPU 处理能力
3. 确认 `max_tokens` 设置差异足够大（短请求和长请求差距越大，效果越明显）
4. 检查显存是否不足导致 SGLang 限制了批处理大小

### 问题 2：部分并发请求超时

**现象：** 部分请求返回超时错误。

**排查步骤：**
1. 增大 OpenAI 客户端的 `timeout` 参数：`client = OpenAI(..., timeout=120)`
2. 减少同时并发的请求数量
3. 降低 `max_tokens` 以缩短单个请求的处理时间
4. 使用 `--mem-fraction-static 0.7` 配合 `--context-length 2048` 优化服务端显存分配

---

### 结语

连续批处理是 SGLang 提升服务吞吐量的核心技术之一。它让 GPU 资源得到更充分的利用，尤其在请求长度差异较大的场景下效果显著。作为用户，你只需正常提交请求，SGLang 会自动完成批次调度优化。